# Machine Learning Practical Assignment
## Data Preprocessing & Feature Selection
### Adult Census Income Dataset

**Group Members**

| Student | Roll Number |
|---|---:|
| Sachin Kumar | 037 |
| Rohit Singh | 036 |
| Shubham Kumar Gupta | 043 |
| Jay Singh | 018 |

### Project Philosophy
**Understand → Calculate → Code → Verify → Interpret**

This notebook is designed around the requirements of the supplied practical-assignment PDF.

# 1. Problem Definition

### Real-world problem
The Adult Census Income dataset represents a classification problem in which demographic, educational, occupational and financial characteristics are used to study whether an individual's annual income falls above or below a specified income threshold.

### Objective
Our objective is to:
- understand and clean the dataset,
- implement important preprocessing techniques from scratch,
- compare manual calculations with library implementations,
- study relationships between features and the target,
- select useful features while avoiding data leakage.

### Target
`income`

### Problem Type
**Binary Classification**

# 2. Dataset Understanding

The Adult/Census Income dataset contains demographic, educational, occupational and financial variables.

The raw UCI version uses the following fields:

`age, workclass, fnlwgt, education, education-num, marital-status, occupation, relationship, race, sex, capital-gain, capital-loss, hours-per-week, native-country, income`

The target is `income`.

> **Important:** We will inspect the raw data before deciding which columns to remove. No feature will be removed merely because it is inconvenient.

In [ ]:
import os, sys, math, urllib.request, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append("../src")

from preprocessing import *
from feature_selection import *

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

print("Libraries imported successfully.")

# 3. Load the Dataset

The notebook can download the UCI Adult data automatically.

The data files are downloaded from the UCI Machine Learning Repository. If `adult.csv` is already placed in `dataset/`, that local file will be used instead.

In [ ]:
DATA_DIR = "../dataset"
os.makedirs(DATA_DIR, exist_ok=True)

csv_path = os.path.join(DATA_DIR, "adult.csv")

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
else:
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
    columns = [
        "age","workclass","fnlwgt","education","education-num","marital-status",
        "occupation","relationship","race","sex","capital-gain","capital-loss",
        "hours-per-week","native-country","income"
    ]
    raw = pd.read_csv(
        url, header=None, names=columns, na_values=[" ?", "?"],
        skipinitialspace=True
    )
    raw.to_csv(csv_path, index=False)
    df = raw.copy()

print("Shape:", df.shape)
df.head()

# 4. Initial Data Exploration

We inspect:
- first records,
- last records,
- shape,
- column names,
- data types,
- unique values,
- missing values,
- descriptive statistics.

In [ ]:
print("First five records:")
display(df.head())

print("Last five records:")
display(df.tail())

print("Shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nUnique values:")
display(df.nunique().to_frame("unique_values"))

print("\nMissing values:")
missing = pd.DataFrame({
    "Missing Values": df.isna().sum(),
    "Missing %": df.isna().mean()*100
})
display(missing)

print("\nDescriptive statistics:")
display(df.describe(include="all").T)

# 5. Basic Statistics — From Scratch

For a numerical feature, we calculate mean, median, mode, variance and standard deviation manually and then verify the results with Pandas.

The population variance used for this assignment is:

\[
Var(X)=\frac{1}{n}\sum_{i=1}^{n}(x_i-\bar{x})^2
\]

In [ ]:
numeric_demo = "age"
values = df[numeric_demo].dropna()

manual_stats = {
    "Mean": manual_mean(values),
    "Median": manual_median(values),
    "Mode": manual_mode(values),
    "Variance": manual_variance(values),
    "Standard Deviation": manual_std(values),
    "Minimum": values.min(),
    "Maximum": values.max(),
    "Range": values.max()-values.min()
}
display(pd.Series(manual_stats, name="Manual"))

verification = {
    "Mean": values.mean(),
    "Median": values.median(),
    "Mode": values.mode().iloc[0],
    "Variance": values.var(ddof=0),
    "Standard Deviation": values.std(ddof=0),
    "Minimum": values.min(),
    "Maximum": values.max(),
    "Range": values.max()-values.min()
}
display(pd.Series(verification, name="Pandas"))

print("Interpretation: the manual and library results should agree up to normal floating-point precision.")

# 6. Missing Value Analysis and Treatment

We first quantify missing values. For the primary implementation we use basic Python/Pandas logic instead of `SimpleImputer`.

Decision rule:
- numerical variables: inspect distribution; use median when outliers/skewness make it more robust,
- categorical variables: use mode when the missing proportion is small,
- a feature with excessive missingness may be removed only with a clear justification.

In [ ]:
before_missing = df.isna().sum()

# Work on a copy
clean_df = df.copy()

# Manual-style treatment
for col in clean_df.columns:
    if clean_df[col].isna().sum() == 0:
        continue
    if pd.api.types.is_numeric_dtype(clean_df[col]):
        fill_value = manual_median(clean_df[col].dropna())
    else:
        fill_value = manual_mode(clean_df[col].dropna())
    clean_df[col] = clean_df[col].fillna(fill_value)

after_missing = clean_df.isna().sum()

comparison = pd.DataFrame({
    "Before Missing": before_missing,
    "After Missing": after_missing
})
display(comparison)

print("Interpretation: missing values are treated using rules based on variable type and distribution rather than blindly deleting records.")

# 7. Duplicate and Invalid Data

We inspect duplicate observations and basic logical inconsistencies.

For this dataset, examples include:
- negative age,
- negative hours worked,
- impossible values in count-like variables,
- inconsistent whitespace/capitalization in categorical fields.

In [ ]:
duplicate_count = clean_df.duplicated().sum()
print("Original records:", len(clean_df))
print("Duplicate records:", duplicate_count)

clean_df = clean_df.drop_duplicates().copy()

# Basic validity checks
invalid_age = (clean_df["age"] <= 0).sum()
invalid_hours = (clean_df["hours-per-week"] < 0).sum()
invalid_capital_gain = (clean_df["capital-gain"] < 0).sum()
invalid_capital_loss = (clean_df["capital-loss"] < 0).sum()

print("\nInvalid-value checks:")
print("Age <= 0:", invalid_age)
print("Hours-per-week < 0:", invalid_hours)
print("Capital-gain < 0:", invalid_capital_gain)
print("Capital-loss < 0:", invalid_capital_loss)

# Normalize categorical whitespace/case
categorical_cols = clean_df.select_dtypes(include="object").columns.tolist()
for col in categorical_cols:
    clean_df[col] = clean_category(clean_df[col])

print("\nRecords after duplicate treatment:", len(clean_df))

# 8. Categorical Encoding — From Scratch

### Label Encoding
Label encoding maps categories to integer codes. It is suitable when the categories have a meaningful order or when the encoded representation is being used carefully.

### One-Hot Encoding
One-hot encoding creates a separate binary column for each category and avoids imposing an artificial numerical order on nominal categories.

We implement both approaches with dictionaries/basic logic first.

In [ ]:
# Label encoding demonstration on sex
label_encoded_sex, sex_mapping = manual_label_encode(clean_df["sex"])
print("Label mapping:", sex_mapping)
display(pd.DataFrame({"sex": clean_df["sex"].head(10), "encoded": label_encoded_sex.head(10)}))

# One-hot demonstration on workclass
one_hot_demo = manual_one_hot(clean_df["workclass"])
display(one_hot_demo.head())

# 9. Outlier Detection — IQR

For a numerical feature:

\[
IQR=Q3-Q1
\]

\[
Lower=Q1-1.5(IQR)
\]

\[
Upper=Q3+1.5(IQR)
\]

We will not automatically delete outliers. We first determine whether they are errors or valid extreme observations.

In [ ]:
outlier_feature = "capital-gain"
q1, q3, iqr, lower, upper = manual_iqr_bounds(clean_df[outlier_feature])

iqr_outliers = clean_df[(clean_df[outlier_feature] < lower) | (clean_df[outlier_feature] > upper)]

print("Feature:", outlier_feature)
print("Q1:", q1)
print("Q3:", q3)
print("IQR:", iqr)
print("Lower bound:", lower)
print("Upper bound:", upper)
print("Detected outliers:", len(iqr_outliers))

# 10. Outlier Detection — Z-Score

\[
z=\frac{x-\mu}{\sigma}
\]

A common rule for potential extreme observations is:

\[
|z|>3
\]

IQR and Z-score can produce different results because IQR is based on quartiles and is more robust to extreme values, while the Z-score depends directly on mean and standard deviation.

In [ ]:
z_values = manual_z_scores(clean_df[outlier_feature].astype(float))
z_outliers = clean_df[np.abs(z_values) > 3]

print("Z-score outliers:", len(z_outliers))
print("IQR outliers:", len(iqr_outliers))

# We retain valid extremes rather than blindly deleting them.
print("Decision: inspect domain validity before removing mathematical outliers.")

# 11. Data Transformation

We investigate skewness and use a log transformation where appropriate.

For non-negative monetary/count variables, `log1p(x) = log(1+x)` is safer than `log(x)` when zeros exist.

We compare the distribution before and after transformation.

In [ ]:
transform_feature = "capital-gain"

fig = plt.figure(figsize=(8,5))
plt.hist(clean_df[transform_feature], bins=40)
plt.title(f"Distribution Before Log Transformation: {transform_feature}")
plt.xlabel(transform_feature)
plt.ylabel("Frequency")
plt.show()

clean_df[transform_feature + "_log"] = np.log1p(clean_df[transform_feature])

fig = plt.figure(figsize=(8,5))
plt.hist(clean_df[transform_feature + "_log"], bins=40)
plt.title(f"Distribution After Log Transformation: {transform_feature}")
plt.xlabel("log1p(" + transform_feature + ")")
plt.ylabel("Frequency")
plt.show()

print("Skewness before:", clean_df[transform_feature].skew())
print("Skewness after:", clean_df[transform_feature + "_log"].skew())

# 12. Feature Scaling — Min-Max Normalization

\[
X' = \frac{X-X_{min}}{X_{max}-X_{min}}
\]

The primary implementation is performed manually. We then verify the result with a library implementation.

In [ ]:
scale_feature = "age"
age_values = clean_df[scale_feature].astype(float).tolist()

manual_norm = manual_minmax(age_values)
norm_series = pd.Series(manual_norm, index=clean_df.index, name="manual_normalized_age")

display(pd.DataFrame({
    "Original": clean_df[scale_feature].head(10),
    "Min": clean_df[scale_feature].min(),
    "Max": clean_df[scale_feature].max(),
    "Normalized": norm_series.head(10)
}))

print("Minimum normalized value:", min(manual_norm))
print("Maximum normalized value:", max(manual_norm))

# 13. Feature Scaling — Standardization

\[
Z=\frac{X-\mu}{\sigma}
\]

The mean and standard deviation are calculated manually.

**Data-leakage rule:** when building the final ML pipeline, these parameters must be learned from the training set and then applied to the test set.

In [ ]:
standardized, train_mean_demo, train_std_demo = manual_standardize(age_values)
display(pd.DataFrame({
    "Original": clean_df[scale_feature].head(10),
    "Standardized": pd.Series(standardized, index=clean_df.index).head(10)
}))

print("Manual mean:", train_mean_demo)
print("Manual standard deviation:", train_std_demo)

# 14. Data Visualization

Required meaningful visualizations:
- histogram,
- box plot,
- bar chart,
- scatter plot,
- correlation heatmap.

Every graph should have a title, axis labels and an interpretation.

In [ ]:
# Histogram
fig = plt.figure(figsize=(8,5))
plt.hist(clean_df["age"], bins=30)
plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Frequency")
plt.show()

# Box plot
fig = plt.figure(figsize=(8,5))
plt.boxplot(clean_df["hours-per-week"])
plt.title("Hours Per Week — Box Plot")
plt.ylabel("Hours Per Week")
plt.show()

# Bar chart
income_counts = clean_df["income"].value_counts()
fig = plt.figure(figsize=(7,5))
plt.bar(income_counts.index.astype(str), income_counts.values)
plt.title("Income Class Distribution")
plt.xlabel("Income Class")
plt.ylabel("Number of Records")
plt.show()

# Scatter plot
fig = plt.figure(figsize=(8,5))
plt.scatter(clean_df["age"], clean_df["hours-per-week"], alpha=0.2)
plt.title("Age vs Hours Per Week")
plt.xlabel("Age")
plt.ylabel("Hours Per Week")
plt.show()

# Correlation heatmap using Matplotlib
numeric_df = clean_df.select_dtypes(include=np.number)
corr = numeric_df.corr()
fig = plt.figure(figsize=(9,7))
plt.imshow(corr, aspect="auto")
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr.columns)), corr.columns, rotation=90)
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Numerical Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

# 15. Train-Test Split — From Scratch

We randomly shuffle row indices and divide them into training and testing sets.

An 80/20 split is used.

The same observations must not be used both for training and final evaluation.

In [ ]:
# Convert target to a clean binary representation
model_df = clean_df.copy()
model_df["target"] = (model_df["income"].str.contains(">50k")).astype(int)

train_df, test_df = manual_train_test_split(model_df, test_size=0.20, seed=42)

print("Training records:", len(train_df))
print("Testing records:", len(test_df))
print("Training proportion:", len(train_df)/len(model_df))
print("Testing proportion:", len(test_df)/len(model_df))

# 16. Data Leakage Demonstration

### Incorrect
Entire dataset → calculate preprocessing parameters → transform → split

This allows information from the test set to influence preprocessing.

### Preferred
Dataset → split → learn parameters from training set → transform training set → transform test set using the same training parameters.

The same principle applies to:
- imputation,
- normalization,
- standardization,
- supervised feature selection,
- category mappings where learned from data.

In [ ]:
# Demonstration with age
train_age = train_df["age"].astype(float)
test_age = test_df["age"].astype(float)

mu_train = manual_mean(train_age)
sd_train = manual_std(train_age)

train_z = (train_age - mu_train) / sd_train
test_z = (test_age - mu_train) / sd_train

print("Training mean:", mu_train)
print("Training SD:", sd_train)
print("Test transformation uses TRAINING mean and SD.")
display(pd.DataFrame({
    "train_example": train_z.head().values
}))

# 17. Preprocessing Pipeline

Our logical sequence is:

**Raw Dataset**
→ Understand Dataset
→ Initial Exploration
→ Train-Test Split
→ Learn Training Parameters
→ Missing Value Treatment
→ Duplicate/Invalid Data Treatment
→ Encoding
→ Outlier Analysis/Treatment
→ Transformation
→ Scaling
→ Feature Selection
→ Final ML-Ready Dataset

The key reason for this sequence is to prevent information from the test set leaking into learned preprocessing or supervised feature-selection decisions.

# 18. Feature Selection — Variance

A zero-variance feature has the same value for every observation and therefore provides no discriminatory variation.

We calculate variance manually and inspect low-variance numerical variables.

In [ ]:
variance_results = []
for col in model_df.select_dtypes(include=np.number).columns:
    if col == "target": 
        continue
    variance_results.append([col, manual_variance(model_df[col])])

variance_table = pd.DataFrame(variance_results, columns=["Feature","Manual Variance"])
display(variance_table.sort_values("Manual Variance"))

# 19. Feature Selection — Pearson Correlation

Pearson correlation measures the strength and direction of a **linear** relationship.

A low Pearson correlation does not prove that no relationship exists; nonlinear relationships may still be present.

In [ ]:
pearson_results = []
for col in model_df.select_dtypes(include=np.number).columns:
    if col == "target":
        continue
    r = manual_pearson(model_df[col], model_df["target"])
    pearson_results.append([col, r])

pearson_table = pd.DataFrame(pearson_results, columns=["Feature","Pearson r"])
display(pearson_table.sort_values("Pearson r", key=lambda s: s.abs(), ascending=False))

# 20. Feature Selection — Chi-Square

Chi-Square is appropriate for studying the relationship between categorical/discrete features and a categorical target.

\[
\chi^2 = \sum \frac{(O-E)^2}{E}
\]

\[
E=\frac{RowTotal\times ColumnTotal}{GrandTotal}
\]

\[
df=(r-1)(c-1)
\]

We manually construct contingency tables and calculate expected frequencies and contributions.

In [ ]:
categorical_candidates = [
    c for c in ["workclass","education","marital-status","occupation",
                "relationship","race","sex","native-country"]
    if c in model_df.columns
]

chi_results = []
for col in categorical_candidates:
    table = contingency_table(model_df[col], model_df["target"])
    chi2, df_chi, expected, contributions = manual_chi_square(table)
    chi_results.append([col, chi2, df_chi])

chi_table = pd.DataFrame(chi_results, columns=["Feature","Chi-Square","df"])
display(chi_table.sort_values("Chi-Square", ascending=False))

# Show one complete manual example
example_feature = categorical_candidates[0]
example_table = contingency_table(model_df[example_feature], model_df["target"])
chi2, df_chi, expected, contributions = manual_chi_square(example_table)
print("Example feature:", example_feature)
print("Observed:")
display(example_table)
print("Expected:")
display(expected)
print("Contributions:")
display(contributions)
print("Total Chi-Square:", chi2, "Degrees of freedom:", df_chi)

# 21. Feature Selection — ANOVA F-Test

ANOVA compares a numerical feature across categorical target groups.

We calculate:
- overall mean,
- group means,
- between-group variation,
- within-group variation,
- degrees of freedom,
- F-statistic.

A relatively high F-statistic indicates that group means differ substantially relative to within-group variation.

In [ ]:
anova_results = []

for col in model_df.select_dtypes(include=np.number).columns:
    if col == "target":
        continue
    groups = [
        model_df.loc[model_df["target"] == 0, col].dropna().to_numpy(),
        model_df.loc[model_df["target"] == 1, col].dropna().to_numpy()
    ]
    result = manual_anova(groups)
    anova_results.append([col, result["f_statistic"], result["grand_mean"]])

anova_table = pd.DataFrame(anova_results, columns=["Feature","F-statistic","Grand Mean"])
display(anova_table.sort_values("F-statistic", ascending=False))

# 22. Feature Selection — Mutual Information

For discrete/categorical variables:

\[
H(Y)=-\sum P(y)\log_2 P(y)
\]

\[
MI(X;Y)=H(Y)-H(Y|X)
\]

Mutual information measures shared information and can capture more general statistical dependence than Pearson correlation.

For this assignment, the manual calculation is demonstrated on categorical/discrete data rather than pretending an arbitrary frequency table is an exact estimator for continuous variables.

In [ ]:
mi_results = []

for col in categorical_candidates:
    mi = manual_mutual_information(model_df[col], model_df["target"])
    mi_results.append([col, mi])

mi_table = pd.DataFrame(mi_results, columns=["Feature","Manual MI (bits)"])
display(mi_table.sort_values("Manual MI (bits)", ascending=False))

# One complete entropy/MI demonstration
mi_demo_feature = categorical_candidates[0]
print("Target entropy H(Y):", entropy(model_df["target"]))
print("Manual MI for", mi_demo_feature, ":", manual_mutual_information(model_df[mi_demo_feature], model_df["target"]))

# 23. Feature-Selection Method Comparison

| Method | Feature Type | Target Type | Main Purpose |
|---|---|---|---|
| Variance | Numerical | Not required | Detect low-variance features |
| Pearson | Numerical | Numerical/binary encoded | Linear relationship |
| Chi-Square | Categorical/Discrete | Categorical | Statistical dependence |
| ANOVA F-Test | Numerical | Categorical | Difference across groups |
| Mutual Information | Discrete/Categorical for manual calculation | Discrete/Categorical | Shared information/dependence |

**Important:** We do not apply every technique blindly. The technique is chosen according to the data type and the question being studied.

# 24. Final Feature Selection Decision

The final decision should combine:
- statistical evidence,
- data type,
- redundancy,
- missingness,
- interpretability,
- domain meaning,
- and whether the feature is available at prediction time.

Do not remove a feature merely because one score is small.

The following table is generated as a starting point; the final Keep/Remove decisions should be reviewed by the group after seeing the actual results.

In [ ]:
# Compact evidence tables
display(variance_table.sort_values("Manual Variance"))
display(pearson_table.sort_values("Pearson r", key=lambda s: s.abs(), ascending=False))
display(chi_table.sort_values("Chi-Square", ascending=False))
display(anova_table.sort_values("F-statistic", ascending=False))
display(mi_table.sort_values("Manual MI (bits)", ascending=False))

print("Use these results to write the final Keep/Remove decision with a specific justification for every removal.")

# 25. Before vs After Summary

Fill this table after the final preprocessing and feature-selection decisions are confirmed.

| Parameter | Before | After |
|---|---:|---:|
| Records | | |
| Features | | |
| Missing Values | | |
| Duplicate Records | | |
| Categorical Features | | |
| Outliers | | |
| Selected Features | | |

### Interpretation
Explain how preprocessing improved data quality and how feature selection reduced redundancy or dimensionality while retaining useful information.

In [ ]:
summary = {
    "Original Records": len(df),
    "Current Records": len(clean_df),
    "Original Features": len(df.columns)-1,
    "Current Columns": len(clean_df.columns),
    "Remaining Missing Values": int(clean_df.isna().sum().sum()),
    "Duplicate Records Removed": int(duplicate_count),
}
display(pd.Series(summary, name="Value"))

# 26. Library Verification

After the from-scratch calculations, library implementations may be used to verify selected results.

The goal is not to replace the manual implementation. It is to demonstrate that the underlying logic agrees with established implementations.

In [ ]:
# Verification examples
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.feature_selection import chi2, mutual_info_classif
from scipy.stats import pearsonr, f_oneway

# Min-Max verification
scaler = MinMaxScaler()
lib_norm = scaler.fit_transform(clean_df[["age"]]).ravel()
print("Max absolute difference, manual vs MinMaxScaler:",
      np.max(np.abs(np.array(manual_norm) - lib_norm)))

# Standardization verification
std_scaler = StandardScaler()
lib_z = std_scaler.fit_transform(clean_df[["age"]]).ravel()
print("Max absolute difference, manual vs StandardScaler:",
      np.max(np.abs(np.array(standardized) - lib_z)))

# Pearson verification
r_manual = manual_pearson(model_df["age"], model_df["target"])
r_library, p_library = pearsonr(model_df["age"], model_df["target"])
print("Manual Pearson:", r_manual)
print("Library Pearson:", r_library)
print("p-value:", p_library)

# ANOVA verification
g0 = model_df.loc[model_df["target"]==0, "age"].dropna()
g1 = model_df.loc[model_df["target"]==1, "age"].dropna()
f_library, p_anova = f_oneway(g0, g1)
print("Manual ANOVA F:", manual_anova([g0.to_numpy(), g1.to_numpy()])["f_statistic"])
print("Library ANOVA F:", f_library)
print("ANOVA p-value:", p_anova)

# 27. Optional Model-Based Validation

This section is optional according to the assignment.

If used, compare:
- model with all suitable features,
- model with selected features.

Do not focus only on accuracy. Discuss dimensionality, redundancy, interpretability, training time and whether performance was maintained or improved.

In [ ]:
# Optional extension:
# Build a simple Logistic Regression model after the final feature-selection
# decisions have been reviewed by the group.
#
# Keep this section after the preprocessing/feature-selection work so that
# the assignment remains focused on understanding preprocessing and selection.

# 28. Final Conclusions

### Key Findings
Write 5–8 concise findings after running the notebook, for example:
1. Which columns contained missing values?
2. Which imputation strategy was selected and why?
3. Which variables showed notable outliers?
4. Which transformations improved skewness?
5. Which features were strongly associated with the target?
6. Which features were redundant?
7. Which features were finally selected?
8. How many features were removed?

### Final Selected Features
List the final features here.

### Final Remarks
The project demonstrates that preprocessing and feature selection should be based on the data type, statistical reasoning and domain context rather than blindly applying library functions.